# Exercise 1:
### Convert `sales.csv` file into JSON and Parquet formats

In [1]:
import pandas as pd
import json

# CSV to JSON:
def csv_to_json(input_file, output_file):
    try:
        # Read the input CSV file
        df = pd.read_csv(input_file)

        # Convert DataFrame to JSON and save to a file 
        df.to_json(output_file, orient='records', indent=4)
        
        print(f"File successfully converted to JSON: {output_file}")
    except Exception as e:
        print(f"Error during conversion: {e}")

# Run the function
csv_to_json('datasets/sales.csv', 'datasets/sales.json')

# JSON to Parquet
def json_to_parquet(input_file, output_file):
    try:
        # Read the input JSON file
        df = pd.read_json(input_file)

        # Save DataFrame as Parquet file 
        df.to_parquet(output_file, engine='pyarrow', index=False)
        
        print(f"File successfully converted to Parquet: {output_file}")
    except Exception as e:
        print(f"Error during conversion: {e}")

# Run the function
json_to_parquet('datasets/sales.json', 'datasets/sales.parquet')

File successfully converted to JSON: datasets/sales.json
File successfully converted to Parquet: datasets/sales.parquet


# Exercise 2:
### Load `sales` data into a PostgreSQL database
- Handle missing or corrupt records gracefully
- Write queries to verify data integrity post-load

## PostgreSQL Database Connection

This section loads the database credentials from a `.env` file and establishes a connection to PostgreSQL.

Using environment variables is considered a security best practice because sensitive information such as usernames and passwords is not stored directly in the notebook or committed to GitHub.

A connection test is performed before loading any data to ensure the database is accessible.

In [2]:
import os
import psycopg2
from dotenv import load_dotenv

load_dotenv()

db_config = {
    "dbname": os.getenv("DB_NAME"),
    "user": os.getenv("DB_USER"),
    "password": os.getenv("DB_PASSWORD"),
    "host": os.getenv("DB_HOST"),
    "port": os.getenv("DB_PORT")
}

# Before loading the CSV, test the connection
try:
    conn = psycopg2.connect(**db_config)
    print("Successfully connected to PostgreSQL!")
    conn.close()

except Exception as e:
    print(f"Connection failed: {e}")

Successfully connected to PostgreSQL!


## Load CSV Data into PostgreSQL

This function:

1. Reads the CSV file into a pandas DataFrame.
2. Connects to PostgreSQL using the credentials stored in the `.env` file.
3. Inserts each record into the target table.
4. Commits the transaction once all records have been loaded.
5. Handles any errors gracefully and closes the database connection.

This approach is suitable for learning purposes and small datasets. For larger datasets, PostgreSQL's `COPY` command would provide significantly better performance.

In [3]:
def load_csv_to_postgres(input_file, table_name):

    try:
        # Establish database connection
        conn = psycopg2.connect(**db_config)
        cursor = conn.cursor()
        
        # Read CSV into a DataFrame
        df = pd.read_csv(input_file)

        columns = ",".join(df.columns)
        placeholders = ",".join(["%s"] * len(df.columns))

        # Load data to PostgreSQL table
        query = f"""
            INSERT INTO {table_name} ({columns})
            VALUES ({placeholders})
        """

        for _, row in df.iterrows():
            cursor.execute(query, tuple(row))

        conn.commit()
        print(f"Data successfully loaded into the '{table_name}' table.")

    except Exception as e:
        print(f"Error during data loading: {e}")

    finally:
        cursor.close()
        conn.close()

## Execute Data Load

The function below loads the contents of `sales.csv` into the PostgreSQL `sales` table.

After execution, validation queries should be run to verify that all records were loaded correctly and that no data integrity issues exist.

In [4]:
load_csv_to_postgres("datasets/sales.csv", "sales_data")

Data successfully loaded into the 'sales_data' table.


## Data Integrity Verification

After loading the CSV data into PostgreSQL, it is important to verify that the load completed successfully and that the data remains valid.

The following validation queries will be used to:

1. Confirm that records were loaded into the target table.
2. Verify the total number of records loaded.
3. Check for missing (NULL) values.
4. Identify invalid quantities.
5. Identify invalid prices.

These checks help ensure the quality and integrity of the data after the load process.

To complete these checks **SQLAlchemy** will be used.

While psycopg2 was used to load the data into PostgreSQL, SQLAlchemy is used for the validation queries.

SQLAlchemy provides a higher-level interface for database connectivity and is the recommended approach when working with pandas' `read_sql()` function.

In [5]:
from sqlalchemy import create_engine

# Create a SQLAlchemy engine for running validation queries
engine = create_engine(
    f"postgresql+psycopg2://{db_config['user']}:{db_config['password']}@"
    f"{db_config['host']}:{db_config['port']}/{db_config['dbname']}"
)

In [6]:
# Query 1: Verify data was loaded

query = """
SELECT * FROM sales_data
LIMIT 10;
"""

df = pd.read_sql(query, engine)
print(df)

   id   sale_date product_name  quantity   price
0   1  2025-02-02      Monitor         3   841.0
1   2  2025-10-10       Laptop         3   533.0
2   3  2025-08-27       Laptop         9   802.0
3   4  2025-06-10       Tablet         2   950.0
4   5  2025-06-08     Keyboard         4   355.0
5   6  2025-11-10        Phone         7  1454.0
6   7  2025-02-01       Laptop         2   507.0
7   8  2025-09-12       Tablet         9   700.0
8   9  2025-03-15       Laptop         9   387.0
9  10  2025-02-04      Monitor        10  1995.0


In [7]:
# Query 2: Verify Record Count

query = """
SELECT COUNT(*) AS total_records
FROM sales_data;
"""

df = pd.read_sql(query, engine)
print(df)

   total_records
0         750000


In [8]:
# Query 3: Check for a missing value

query = """
SELECT * FROM sales_data
WHERE sale_date IS NULL
   OR product_name IS NULL
   OR quantity IS NULL
   OR price IS NULL;
"""

df = pd.read_sql(query, engine)
print(df)

Empty DataFrame
Columns: [id, sale_date, product_name, quantity, price]
Index: []


In [9]:
# Query 4: Check for invalid quantities

query = """
SELECT * FROM sales_data
WHERE quantity <= 0;
"""

df = pd.read_sql(query, engine)
print(df)

Empty DataFrame
Columns: [id, sale_date, product_name, quantity, price]
Index: []


In [10]:
# Query 5: Check for invalid prices

query = """
SELECT * FROM sales_data
WHERE price <= 0;
"""

df = pd.read_sql(query, engine)
print(df)

Empty DataFrame
Columns: [id, sale_date, product_name, quantity, price]
Index: []


### Summary

The sales dataset was successfully loaded into PostgreSQL using Python, pandas and psycopg2.

Post-load validation was performed using SQLAlchemy and pandas to verify data integrity.

Validation checks confirmed:

- Records were loaded successfully.
- The expected number of records exists in the target table.
- No missing values were detected.
- No invalid quantities were detected.
- No invalid prices were detected.

These validation steps provide confidence that the dataset was loaded accurately and maintains the expected level of data quality.

# Exercise 3:
### Data validation
- Create data validation for a given file (e.g., checking schema compliance, missing values)

In [5]:
import pandas as pd

def validate_csv(input_file, schema):
    try:
        # Read the CSV file
        df = pd.read_csv(input_file)

        # Check column compliance
        missing_columns = [col for col in schema.keys() if col not in df.columns]
        if missing_columns:
            print(f"Missing columns: {missing_columns}")

        # Validate data types
        for col, dtype in schema.items():
            if col in df.columns and df[col].dtype != dtype:
                print(f"Column '{col}' has incorrect data type. Expected: {dtype}, Found: {df[col].dtype}")

        # Check for missing values
        missing_data = df.isnull().sum()
        print("Missing values in each column:")
        print(missing_data[missing_data>0])
    except Exception as e:
        print(f"Error during validation: {e}")

# Define schema as expected columns and data types
expected_schema = {
    'id':'int64',
    'sale_date':'object',
    'product_name':'object',
    'quantity':'int64',
    'price':'int64'
}

# Validate file
validate_csv("datasets/sales.csv", expected_schema)

Missing values in each column:
Series([], dtype: int64)
